# Leakage-aware statistical data cleaning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/statistical-data-cleaning.ipynb)

Companion notebook for the ReliableML paper `docs/latex/statistical-data-cleaning.tex`. It demonstrates `StatisticalDataCleaner` / `clean_statistical_data`: fit-time imputation rules, IQR outlier diagnostics, the three intensities, and fold-safe use in a pipeline.

All numbers shown below are computed by the cells themselves (fixed random seeds); nothing is hard-coded. Run the cells top to bottom.

## 0. Setup (parameters)

**Parameter cell - choose ONE installation source.** The default installs the released package from PyPI. To try the unreleased development version, comment out option A and uncomment option B. If neither works (for example a restricted network), run the fallback cell below it and upload a zip of the repository.

In [ ]:
#@title PARAMETERS: install ReliableML
# ---- Option A (default): released version from PyPI ----
!pip install -q ReliableML
# ---- Option B: unreleased version from GitHub (uncomment; comment option A) ----
# !pip install -q "git+https://github.com/Jabualani/ML-Reliabilty.git"
# Plots need matplotlib (preinstalled on Colab); locally: pip install matplotlib

In [ ]:
#@title Fallback: upload a repository zip (only runs if ml_reliability is not importable)
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("ml_reliability") is None:
    from google.colab import files  # Colab only

    # GitHub: Code > Download ZIP, then pick the file in the dialog
    uploaded = files.upload()
    for archive in uploaded:
        zipfile.ZipFile(archive).extractall("repo_zip")
    package_init = next(pathlib.Path("repo_zip").rglob("ml_reliability/__init__.py"))
    sys.path.insert(0, str(package_init.parent.parent))
    # the repo's dependencies (numpy, pandas, scikit-learn, scipy, statsmodels) are
    # preinstalled on Colab; otherwise: !pip install -q numpy pandas scikit-learn scipy statsmodels

import ml_reliability
print("ml_reliability loaded from:", pathlib.Path(ml_reliability.__file__).parent)

## 1. Rules implemented

For a numeric feature $X_j$ let $\mathcal O_j$ be its observed values.

* **Binary** (at most two distinct observed values): impute the observed **mode**.
* **Approximately normal** (heuristic, *not* a hypothesis test): at least 10 observations, $|\text{mean}-\text{median}|\le 0.1\,\sigma$ (population SD), $|\text{skew}|<0.5$ and $|\text{excess kurtosis}|<1.5$: impute the **mean** $\hat\mu_j=|\mathcal O_j|^{-1}\sum_{x\in\mathcal O_j}x$.
* **Otherwise**: impute the **median** $\hat m_j=\operatorname{median}(\mathcal O_j)$.
* **Categorical**: observed mode; an all-missing categorical column gets the sentinel `__reliableml_missing__`.
* **All-missing numeric**: a warning, fill with $0$, and the feature is listed as uninformative.

**IQR fences** (Tukey), learned on fit data only:
$$IQR_j=Q_{3j}-Q_{1j},\qquad L_j=Q_{1j}-k\,IQR_j,\qquad U_j=Q_{3j}+k\,IQR_j,$$
with $k=1.5$ (light, moderate) or $k=1.0$ (heavy). **Light** imputes only and *flags* values outside $[L_j,U_j]$; **moderate/heavy** also clip to $[L_j,U_j]$. The bounds are reused unchanged in `transform`.

**Leakage control.** Inside cross-validation the cleaner must be refit on each training fold:
$$\widehat f^{(-v)}=\operatorname{Fit}\!\big(\widehat T^{(-v)}(X^{(-v)}_{\text{train}}),y^{(-v)}_{\text{train}}\big),\qquad \widehat y^{(-v)}_{\text{valid}}=\widehat f^{(-v)}\!\big(\widehat T^{(-v)}(X^{(-v)}_{\text{valid}})\big).$$

## 2. Synthetic data with every case the rules distinguish

In [ ]:
import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ml_reliability import StatisticalDataCleaner, clean_statistical_data

rng = np.random.default_rng(0)
n = 400
df = pd.DataFrame({
    "symmetric": rng.normal(50, 5, n),            # approx. normal -> mean
    "contaminated": rng.normal(50, 5, n),         # same law + injected outliers -> heuristic rejects normality
    "skewed": rng.lognormal(3, 0.8, n),            # right-skewed -> median
    "binary": rng.integers(0, 2, n).astype(float), # <=2 values -> mode
    "city": rng.choice(["north", "south", "west"], n, p=[0.5, 0.3, 0.2]),
    "all_missing": np.full(n, np.nan),             # unidentifiable numeric
})
df.loc[rng.choice(n, 8, replace=False), "contaminated"] = 120.0   # injected outliers
for col in ["symmetric", "contaminated", "skewed", "binary", "city"]:
    df.loc[rng.random(n) < 0.10, col] = np.nan                 # ~10% missing completely at random

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    cleaned, cleaner = clean_statistical_data(df, cleaning_intensity="light", return_cleaner=True)
print("warnings:", [str(w.message) for w in caught])
print("remaining NaNs after cleaning:", int(cleaned.isna().sum().sum()))
pd.DataFrame({"missing at fit": cleaner.missing_counts_, "fill value": cleaner.fill_values_}).astype(str)

In [ ]:
report = cleaner.get_cleaning_report()
for key, value in report.items():
    print(f"{key}: {value}")

observed = df["symmetric"].dropna()
print()
print("symmetric: mean=%.3f median=%.3f -> fill=%.3f (mean rule)" % (observed.mean(), observed.median(), cleaner.fill_values_["symmetric"]))
observed = df["contaminated"].dropna()
print("contaminated: mean=%.3f median=%.3f -> fill=%.3f (heuristic rejects normality -> median)" % (observed.mean(), observed.median(), cleaner.fill_values_["contaminated"]))
observed = df["skewed"].dropna()
print("skewed   : mean=%.3f median=%.3f -> fill=%.3f (median rule)" % (observed.mean(), observed.median(), cleaner.fill_values_["skewed"]))

## 3. Intensity: light only flags, moderate/heavy clip

Observed minimum/maximum of the `contaminated` column (which contains injected extreme values) after each mode, with the fit-time fences $L_j, U_j$.

In [ ]:
rows = []
for intensity in ["light", "moderate", "heavy"]:
    out, fitted = clean_statistical_data(df, cleaning_intensity=intensity, return_cleaner=True)
    low, high = fitted.outlier_bounds_["contaminated"]
    rows.append({"intensity": intensity, "L": round(low, 2), "U": round(high, 2),
                 "min after": round(out["contaminated"].min(), 2), "max after": round(out["contaminated"].max(), 2),
                 "capping applied": fitted.get_cleaning_report()["outlier_capping_applied"]})
display_table = pd.DataFrame(rows)
print(display_table.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharex=True)
for ax, intensity in zip(axes, ["light", "moderate", "heavy"]):
    out, fitted = clean_statistical_data(df, cleaning_intensity=intensity, return_cleaner=True)
    ax.hist(out["contaminated"], bins=30, color="#4c78a8")
    low, high = fitted.outlier_bounds_["contaminated"]
    ax.axvline(low, color="crimson", ls="--"); ax.axvline(high, color="crimson", ls="--")
    ax.set_title(f"{intensity} mode"); ax.set_xlabel("contaminated")
plt.tight_layout(); plt.show()

## 4. Fit-time statistics are reused on new data (no refitting)

The cleaner is fitted on a training split; the test split is imputed with the *training* values, and a batch with a different column order is rejected.

In [ ]:
train, test = df.iloc[:300].copy(), df.iloc[300:].copy()
cleaner_tr = StatisticalDataCleaner().fit(train)
test_clean = cleaner_tr.transform(test)
print("fill value learned on train for 'skewed':", round(cleaner_tr.fill_values_["skewed"], 4))
print("test-set median of 'skewed' (NOT used) :", round(test["skewed"].median(), 4))
filled = test["skewed"].isna()
print("values written into missing test cells:", sorted(set(np.round(test_clean.loc[filled, "skewed"], 4))))
try:
    cleaner_tr.transform(test[test.columns[::-1]])
except ValueError as error:
    print("schema check ->", error)

## 5. Bundled dataset: cleaning inside cross-validation

Breast-cancer features get ~10% missing-completely-at-random values. Placing the cleaner *inside* the scikit-learn `Pipeline` refits it on every training fold; the per-fold imputation value for one feature shows that each fold learns its own statistics.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

bc = load_breast_cancer(as_frame=True)
Xb = bc.data.iloc[:, :10].copy()
yb = bc.target
mask_rng = np.random.default_rng(1)
Xb = Xb.mask(mask_rng.random(Xb.shape) < 0.10)
print("fraction missing:", round(float(Xb.isna().mean().mean()), 3))

pipe = Pipeline([
    ("clean", StatisticalDataCleaner("light")),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
cv = StratifiedKFold(5, shuffle=True, random_state=0)
scores = cross_val_score(pipe, Xb, yb, cv=cv, scoring="accuracy")
print("5-fold accuracy:", np.round(scores, 3), "| mean:", round(scores.mean(), 3))

fold_fill = []
for train_idx, _ in cv.split(Xb, yb):
    fitted = StatisticalDataCleaner().fit(Xb.iloc[train_idx])
    fold_fill.append(fitted.fill_values_["mean radius"])
print("per-fold fill value for 'mean radius':", np.round(fold_fill, 4))

## Interpretation and limitations

**How to read the output.** The cleaning report lists fit-time missing counts and IQR outlier *flags*; a flag is a prompt to investigate, not a deletion instruction. Compare the three intensities to see what capping changes before choosing it. Per-fold fill values differ because each fold fits its own cleaner, which is what keeps validation rows from influencing imputation.

**Limitations.** Single-value imputation understates uncertainty and can distort correlations; the normality decision is a heuristic that is unstable for small samples; IQR fences are univariate and can overflag skewed data; capping compresses legitimate extremes (hence light is the default); zero-fill of an all-missing column is a software fallback, not an estimate. The transformer does no encoding, scaling, multiple imputation or missingness-mechanism modeling.

These notebooks are methodological demonstrations on synthetic data and small bundled scikit-learn datasets. They do not show that any method improves performance on your data; that requires a prespecified evaluation on independent data.